In [ ]:
import pandas as pd
import os

# 1. Define file names (Ensure your file name matches exactly what's uploaded)
input_file = "export_01_Jun_2026_14-13-34.xlsx"
output_file = "Medical_Scoping_Review_Screening.xlsx"

# Quick validation check to make sure you uploaded the file
if not os.path.exists(input_file):
    raise FileNotFoundError(f"⚠️ Could not find '{input_file}' in the Colab directory. Please upload it via the left sidebar folder icon.")

print("🚀 Starting the medical data screening extract...")

# 2. Define the broad medical keyword list for the initial Scoping Review pool
medical_keywords = [
    'medical', 'health', 'hospital', 'casualty', 'evacuation', 'medevac',
    'casevac', 'medcoe', 'wound', 'injury', 'medic', 'doctor', 'nurse',
    'disease', 'patient', 'triage', 'blood', 'trauma', 'clinical', 'pharmaceutical'
]

# Create a regex-ready pipe-separated string of the keywords
keyword_regex = '|'.join(medical_keywords)

# Initialize an empty list to hold our matched dataframes
screening_pool = []

# Load the Excel workbook structure
xls = pd.ExcelFile(input_file)

# 3. Process each sheet systematically
for sheet in xls.sheet_names:
    df = pd.read_excel(input_file, sheet_name=sheet)
    if df.empty:
        continue

    # Generate a boolean mask scanning every column for our medical keywords
    mask = pd.Series([False] * len(df))
    for col in df.columns:
        mask = mask | df[col].astype(str).str.contains(keyword_regex, case=False, na=False)

    matched_df = df[mask].copy()

    if not matched_df.empty:
        # Add metadata indicating the data's origin
        matched_df['Source_Sheet'] = sheet

        # 4. Standardize the disparate text fields into a single 'Main_Text' review column
        if sheet == 'AAR':
            matched_df['Main_Text'] = matched_df['Executive Summary'].astype(str)
        elif sheet == 'BIN':
            matched_df['Main_Text'] = matched_df['Description'].astype(str)
        elif sheet == 'OBS':
            # Combine Observation and Discussion for total context
            matched_df['Main_Text'] = "OBSERVATION: " + matched_df['Observation'].astype(str) + " \n\nDISCUSSION: " + matched_df['Discussion'].astype(str)
        elif sheet == 'PVR':
            matched_df['Main_Text'] = matched_df['Services'].astype(str)
        else:
            # Fallback for COP or other minor sheets
            matched_df['Main_Text'] = matched_df['Title'].astype(str)

        # Isolate key columns for clean, scannable screening workspace
        cols_to_keep = ['Source_Sheet', 'ID', 'Title', 'Created Date', 'Organization', 'Main_Text']
        screening_pool.append(matched_df[cols_to_keep])

# 5. Consolidate into a single master worksheet
final_screening_df = pd.concat(screening_pool, ignore_index=True)

# 6. Inject empty tracking columns for your PRISMA auditing framework
final_screening_df['Screening_Decision'] = ""  # For you to fill: INCLUDE / EXCLUDE / PENDING
final_screening_df['Exclusion_Reason'] = ""   # For you to fill if excluded
final_screening_df['Reviewer_Notes'] = ""

# 7. Write to a fresh Excel file
final_screening_df.to_excel(output_file, index=False)

print(f"✅ Success! Processed {len(final_screening_df)} medical-related records.")
print(f"📂 The new file '{output_file}' is ready. Refresh your left sidebar and download it!")

🚀 Starting the medical data screening extract...


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""C

✅ Success! Processed 190 medical-related records.
📂 The new file 'Medical_Scoping_Review_Screening.xlsx' is ready. Refresh your left sidebar and download it!


In [ ]:
import pandas as pd
import os

# 1. Load the medical screening pool we created earlier
input_file = "Medical_Scoping_Review_Screening (1).xlsx"
output_file = "Medical_Scoping_Review_PCC_Screened.xlsx"

if not os.path.exists(input_file):
    raise FileNotFoundError(f"⚠️ Could not find '{input_file}'. Run the first script first!")

df = pd.read_excel(input_file)
print(f"Loaded {len(df)} records for PCC screening...")

# 2. Define strict PCC Criteria Arrays based on uploaded Table 1

# Population: Ukrainian military service members (Exclude US, civilians, contractors)
pop_include = ['ukraine', 'ukrainian', 'uaf', 'afu', 'armed forces of ukraine']
pop_exclude = ['us military', 'u.s. military', 'dod civilian', 'contractor', 'veteran', 'civilian']

# Concept: DNBI (Disease and Non-Battle Injury) (Exclude traumatic combat injuries)
concept_include = ['dnbi', 'disease', 'illness', 'infection', 'preventable', 'cold weather', 'trench foot', 'hygiene', 'public health', 'non-battle']
concept_exclude = ['shrapnel', 'gunshot', 'gsw', 'blast injury', 'combat trauma', 'animal study', 'bioterrorism']

# Context: Russo-Ukrainian Conflict
context_include = ['russia', 'russian', 'ukraine', 'ukrainian', 'invasion', 'conflict']
context_exclude = ['operation enduring freedom', 'iron swords', 'iraq', 'afghanistan', 'gwot']

# Hard exclusion phrases (Template fillers)
template_exclusions = ['medical: ntr', 'medical: nothing to report']

# 3. Processing Function to evaluate each row
def apply_pcc_framework(row):
    text = str(row['Main_Text']).lower()
    title = str(row['Title']).lower()
    combined_text = text + " " + title

    # Check 1: Date Exclusion (Exclude anything prior to 2020)
    # Note: Assumes 'Created Date' is a datetime object or parseable string
    try:
        created_year = pd.to_datetime(row['Created Date']).year
        if created_year < 2020:
            return "EXCLUDE", "Timespan", f"Created in {created_year} (Prior to 2020 limit)."
    except:
        pass # If date is missing/unparseable, skip this check

    # Check 2: Template Fillers
    if any(phrase in text for phrase in template_exclusions):
        return "EXCLUDE", "Type of Evidence", "Automated rule caught 'MEDICAL: NTR' placeholder."

    # Check 3: Concept Exclusions (Traumatic combat injuries, animal studies)
    found_c_excl = [word for word in concept_exclude if word in combined_text]
    if found_c_excl:
        return "EXCLUDE", "Concept", f"Contains excluded traumatic/animal terms: {found_c_excl}"

    # Check 4: Context Exclusions (Prior conflicts)
    found_ctx_excl = [word for word in context_exclude if word in combined_text]
    if found_ctx_excl:
        return "EXCLUDE", "Context", f"Contains excluded conflict terms: {found_ctx_excl}"

    # Check 5: Population Exclusions (U.S. forces, civilians)
    # *Note: This is tricky in JLLS as reports often mention U.S. forces observing.
    # We will flag it for manual review rather than hard exclusion if U.S. forces are mentioned alongside Ukraine.
    found_p_excl = [word for word in pop_exclude if word in combined_text]

    # Calculate Inclusions
    p_matches = [word for word in pop_include if word in combined_text]
    c_matches = [word for word in concept_include if word in combined_text]
    ctx_matches = [word for word in context_include if word in combined_text]

    notes = f"Matched P: {p_matches} | Matched C: {c_matches} | Matched Context: {ctx_matches}"

    # Decision Logic
    if len(p_matches) > 0 and len(c_matches) > 0 and len(ctx_matches) > 0:
        if found_p_excl:
             return "PENDING", "Population Mixed", f"Passes criteria but contains excluded pop terms ({found_p_excl}). Needs manual review. {notes}"
        return "INCLUDE", "", f"Passes automated PCC threshold for DNBI in Ukraine. {notes}"
    else:
        missing = []
        if not p_matches: missing.append("Population (Ukrainian Mil)")
        if not c_matches: missing.append("Concept (DNBI)")
        if not ctx_matches: missing.append("Context (Russo-UKR Conflict)")

        reason = f"Missing core PCC domains: {', '.join(missing)}"
        return "EXCLUDE", "PCC Missing", f"Automated pass failed criteria. {notes}"

# 4. Apply the framework across the dataframe
print("Applying strict DNBI/Ukraine PCC rules to text fields...")
decisions = df.apply(apply_pcc_framework, axis=1)

# 5. Populate PRISMA columns
df['Screening_Decision'] = [d[0] for d in decisions]
df['Exclusion_Reason'] = [d[1] for d in decisions]
df['Reviewer_Notes'] = [d[2] for d in decisions]

# 6. Save the newly audited spreadsheet
df.to_excel(output_file, index=False)

# Metrics
include_count = (df['Screening_Decision'] == "INCLUDE").sum()
exclude_count = (df['Screening_Decision'] == "EXCLUDE").sum()
pending_count = (df['Screening_Decision'] == "PENDING").sum()

print("\n--- Automated Screening Summary ---")
print(f"✅ Potential INCLUDES: {include_count} rows")
print(f"⚠️ Manual PENDING: {pending_count} rows (Contains mixed population terms)")
print(f"❌ Automated EXCLUDES: {exclude_count} rows")
print(f"📂 Download your audited file: '{output_file}'")

Loaded 190 records for PCC screening...
Applying strict DNBI/Ukraine PCC rules to text fields...

--- Automated Screening Summary ---
✅ Potential INCLUDES: 10 rows
⚠️ Manual PENDING: 5 rows (Contains mixed population terms)
❌ Automated EXCLUDES: 175 rows
📂 Download your audited file: 'Medical_Scoping_Review_PCC_Screened.xlsx'


In [ ]:
%%bash
echo "# Scoping-Review-DNBI" >> README.md
git init
git add README.md
git commit -m "first commit"
git branch -M main
git remote add origin https://github.com/jamesbaldock/Scoping-Review-DNBI.git
git push -u origin main

SyntaxError: invalid syntax (763039924.py, line 1)